In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import json

# Add src to path
sys.path.append('../src')

from audio_processing.pitch_detect import PitchDetector
from audio_processing.smoothing import NoteStabilizer
from feature_extraction.extract_notes import NoteExtractor, RAGA_TEMPLATES
from utils.helpers import print_banner

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load Audio and Detect Pitch

In [ ]:
# Select audio file
DATA_DIR = '../data/raw'
ragas = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]

if ragas:
    sample_raga = ragas[0]
    raga_path = os.path.join(DATA_DIR, sample_raga)
    audio_files = [f for f in os.listdir(raga_path) if f.endswith(('.mp3', '.wav', '.m4a'))]
    
    if audio_files:
        AUDIO_FILE = os.path.join(raga_path, audio_files[0])
        print(f"Analyzing: {AUDIO_FILE}")
        print(f"Expected Raga: {sample_raga}")
    else:
        AUDIO_FILE = None
else:
    AUDIO_FILE = None

In [ ]:
# Detect pitch and extract notes
import librosa

if AUDIO_FILE:
    print("Loading audio...")
    y, sr = librosa.load(AUDIO_FILE, duration=30)
    
    print("Detecting pitch...")
    detector = PitchDetector(sample_rate=sr)
    pitch_contour = detector.detect_pitch_yin(y)
    smoothed_pitch = detector.smooth_pitch(pitch_contour)
    
    print("Estimating tonic...")
    tonic = detector.estimate_tonic(smoothed_pitch)
    print(f"Tonic: {tonic:.2f} Hz")

## 2. Extract Stable Notes

In [ ]:
# Initialize note stabilizer
stabilizer = NoteStabilizer(stable_duration_ms=100, min_note_duration_ms=50)

# Detect stable regions
print("Detecting stable note regions...")
stable_regions = stabilizer.detect_stable_regions(
    smoothed_pitch, 
    hop_length=512, 
    sample_rate=sr
)

print(f"Found {len(stable_regions)} stable note regions")

In [ ]:
# Convert to note sequence with timings
note_sequence = []
frame_duration = 512 / sr

for start_idx, end_idx, mean_pitch in stable_regions:
    # Convert to cents
    cents = 1200 * np.log2(mean_pitch / tonic)
    
    # Get note name
    note_name, deviation = detector.cents_to_note(cents)
    
    # Calculate timing
    start_time = start_idx * frame_duration
    duration = (end_idx - start_idx) * frame_duration
    
    note_sequence.append((note_name, start_time, duration))

print(f"\nExtracted {len(note_sequence)} notes")
print("\nFirst 10 notes:")
for i, (note, start, dur) in enumerate(note_sequence[:10]):
    print(f"  {i+1}. {note:4s} at {start:6.2f}s (duration: {dur:5.3f}s)")

## 3. Extract Arohanam and Avarohanam

In [ ]:
# Initialize note extractor
extractor = NoteExtractor()

# Get just note names
notes = [note for note, _, _ in note_sequence]

# Extract arohanam (ascending)
arohanam = extractor.extract_arohanam(notes)
print("Arohanam (Ascending):")
print(" → ".join(arohanam))

# Extract avarohanam (descending)
avarohanam = extractor.extract_avarohanam(notes)
print("\nAvarohanam (Descending):")
print(" → ".join(avarohanam))

## 4. Analyze Raga Patterns

In [ ]:
# Comprehensive pattern analysis
analysis = extractor.analyze_note_patterns(note_sequence)

print_banner("RAGA PATTERN ANALYSIS")
print(f"\nUnique Notes: {', '.join(analysis['unique_notes'])}")
print(f"Note Count: {analysis['note_count']}")
print(f"Scale Type: {analysis['scale_type']}")
print(f"\nArohanam: {' → '.join(analysis['arohanam'])}")
print(f"Avarohanam: {' → '.join(analysis['avarohanam'])}")

if 'vadi_samvadi' in analysis:
    print(f"\nVadi (Most Emphasized): {analysis['vadi_samvadi'].get('vadi', 'N/A')}")
    print(f"Samvadi (Second Most): {analysis['vadi_samvadi'].get('samvadi', 'N/A')}")

In [ ]:
# Display characteristic phrases
print("\nCharacteristic Phrases (Top 10):")
for i, (phrase, count) in enumerate(list(analysis['characteristic_phrases'].items())[:10], 1):
    print(f"  {i}. {phrase} (appears {count} times)")

## 5. Compare with Known Raga Templates

In [ ]:
# Compare with raga database
print("\nComparing with known ragas...\n")

matches = []
for raga_name, template in RAGA_TEMPLATES.items():
    score = extractor.compare_with_raga_template(
        notes,
        template['arohanam'],
        template['avarohanam']
    )
    matches.append((raga_name, score))

# Sort by similarity
matches.sort(key=lambda x: x[1], reverse=True)

print("Raga Similarity Scores:")
for raga, score in matches:
    bar = '█' * int(score * 20)
    print(f"  {raga:20s} {score:5.2%} {bar}")

## 6. Visualize Note Patterns

In [ ]:
# Visualize note distribution
from collections import Counter

note_counts = Counter(notes)
note_names = list(note_counts.keys())
counts = list(note_counts.values())

plt.figure(figsize=(12, 5))
plt.bar(note_names, counts)
plt.xlabel('Note')
plt.ylabel('Frequency')
plt.title('Note Distribution')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize note sequence over time
note_map = {'S': 0, 'R1': 1, 'R2': 2, 'G2': 3, 'G3': 4, 
            'M1': 5, 'M2': 6, 'P': 7, 'D1': 8, 'D2': 9, 'N2': 10, 'N3': 11}

# Create time series
times = [start for _, start, _ in note_sequence]
note_values = [note_map.get(note, np.nan) for note, _, _ in note_sequence]

plt.figure(figsize=(14, 5))
plt.scatter(times, note_values, alpha=0.6, s=50)
plt.plot(times, note_values, alpha=0.3)
plt.xlabel('Time (s)')
plt.ylabel('Note')
plt.yticks(range(12), list(note_map.keys()))
plt.title('Note Sequence Timeline')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Save Extracted Data

In [ ]:
# Save note sequence to CSV
csv_output = '../data/notes/extracted_notes.csv'
extractor.save_to_csv(note_sequence, csv_output)
print(f"Note sequence saved to: {csv_output}")

# Save analysis to JSON
json_output = '../data/notes/pattern_analysis.json'
analysis_output = {
    **analysis,
    'tonic_hz': float(tonic),
    'audio_file': os.path.basename(AUDIO_FILE),
    'expected_raga': sample_raga,
    'similarity_scores': {raga: float(score) for raga, score in matches}
}
extractor.save_to_json(analysis_output, json_output)
print(f"Pattern analysis saved to: {json_output}")

## Conclusion

This notebook demonstrated:
- Extraction of stable note sequences from audio
- Identification of arohanam and avarohanam patterns
- Analysis of characteristic phrases and patterns
- Comparison with known raga templates
- Visualization of note distributions and timelines

**Next Steps:**
- Use extracted patterns for raga classification
- Train ML models with these features
- Build the complete raga identification pipeline